# Demo - Microsoft Web IQ Customer Management and Search API

## 0. Pre-reqs

You must complete these prerequisites before running the steps in the notebook.

### 0.1 Install notebooks required libraries

In [ ]:
!pip install -r ../requirements.txt

### 0.2 Setup a Python logger to provide details logs for the notebook



In [ ]:
import logging
import sys
import colorlog

def configure_logging(level="ERROR"):
    """This function configures logging for code being run based on the specified level.
    Args:
        level (str): The logging level to use (e.g., "DEBUG", "INFO", "WARNING", "ERROR", "CRITICAL").
    """
    try:
        # Convert the level string to uppercase so it matches what the logging library expects
        logging_level = getattr(logging, level.upper(), None)

        handler = logging.StreamHandler(sys.stdout)

        # Color code the log entries
        handler.setFormatter(colorlog.ColoredFormatter(
            '%(log_color)s%(asctime)s - %(name)s - %(levelname)s - %(message)s',
            log_colors={
                'DEBUG': 'cyan',
                'INFO': 'blue',
                'WARNING': 'yellow',
                'ERROR': 'red',
                'CRITICAL': 'red,bg_white',
            }
        ))
        # Configure root logger to send to stdout
        logging.root.handlers = []
        logging.root.addHandler(handler)
        logging.root.setLevel(logging_level)
    except Exception as e:
        print(f"Failed to set up logging: {e}", file=sys.stderr)

configure_logging("INFO")

logger = logging.getLogger(__name__)
logging.getLogger("azure.core.pipeline.policies.http_logging_policy").setLevel(logging.WARNING)
logging.getLogger("azure.identity").setLevel(logging.WARNING)


### 0.3 Authenticate to Azure CLI

You must authenticate to [Azure CLI](https://learn.microsoft.com/en-us/cli/azure/?view=azure-cli-latest) with a user with sufficient permissions to deploy resources to the subscription.

### 0.4 Create a service principal

The demo will require an Entra ID service principal. The service principal will be granted access to the Microsoft Web IQ Customer Management API and Search API. The [APIs only supports the OAuth client credentials flow](https://learn.microsoft.com/en-us/web-iq/guides/create-scopes#prerequisites) and do not support delegated flow.

In [ ]:
import json
import subprocess
from dotenv import set_key

# Creates the app registration, service principal, and client secret using the signed-in Azure CLI user
result = subprocess.run(
    ["az", "ad", "sp", "create-for-rbac", "--name", "webiq-demo-app", "--output", "json"],
    capture_output=True, text=True, check=True,
)
sp = json.loads(result.stdout)

# Save the credentials to the .env file in the base directory
env_file = "../.env"
set_key(env_file, "WEBIQ_APP_CLIENT_ID", sp["appId"])
set_key(env_file, "WEBIQ_APP_CLIENT_SECRET", sp["password"])

logger.info("Service principal created. Client ID: %s", sp["appId"])


### 0.5 Deploy Terraform resources

The Terraform template in this repository will deploy a resource group and Microsoft Web IQ Account. The Microsoft Web IQ Account is the top-level Azure resource. It is a global resource.

Set the region and tags variables to whatever your preference is.

In [ ]:
import json
import os
import subprocess

region = "eastus2"
tags = {"environment": "demo", "project": "webiq"}

# azurerm needs the subscription ID; use the one the Azure CLI is signed in to
subscription_id = subprocess.run(
    ["az", "account", "show", "--query", "id", "--output", "tsv"],
    capture_output=True, text=True, check=True,
).stdout.strip()

# Terraform reads TF_VAR_* environment variables as input variables (maps are passed as JSON)
env = {
    **os.environ,
    "ARM_SUBSCRIPTION_ID": subscription_id,
    "TF_VAR_region": region,
    "TF_VAR_tags": json.dumps(tags),
}

for args in (["init", "-input=false"], ["apply", "-auto-approve", "-input=false"]):
    result = subprocess.run(
        ["terraform", *args, "-no-color"],
        cwd="../infra", env=env, capture_output=True, text=True,
    )
    print(result.stdout)
    if result.returncode != 0:
        raise RuntimeError(result.stderr)


## 1. Administer access to Microsoft Web IQ

In this step you will perform administrative functions in Microsoft Web IQ.

### 1.1 Grant service principal access to Customer Management API

The Customer Management API is used to create scopes (explained later) and determine which Microsoft Web IQ billing accounts and AppIDs the application has access. This API is only accessible using the client credentials flow so we will grant the necessary permissions to the service principal we recreated earlier.

Granting these permissions requires assigning the CustomerApiAccess role at the data-plane level of Microsoft Web IQ to the service principal. This does not grant access to the service principal to use the Microsoft Web IQ Search API.

To grant this permission you will need to log into the [Microsoft Web IQ Portal](https://webiq.microsoft.ai/profiles) as a user. The user must have at least Contributor on the resource. While there do some to be some permissions related to Microsoft.WebIQ, it's not clear which specific permissions are required. There is no documented programmatic way to do this at this time.

Before the Microsoft Web IQ account resource can be administered in the Microsoft Web IQ Portal, it needs to be added as a resource. To do this you will need to navigate to the Account Settings and click the Add Azure Resource button.

![Add Azure Resource](../images/portal-add-resource.png)

In the pop-up you need to fill in the subscription and resource name of the Microsoft Web IQ Account resource.

![Add Azure Resource pop-up](../images/portal-add-azure-resource-popup.png)

Once complete, navigate back to the Account Settings link and click the Manage Access button.

![Select Managed Access](../images/portal-manage-access.png)

In the Manage Access menu the applications that have been granted access to the Customer Management API will be listed. Now add a new application by clicking the Add Application button. Adding an application grants the service principal the CustomerApiAccess data-plane role on the billing account.

In the pop-up enter a display name for the application, the Entra ID tenant ID the service principal is deployed in and the application id.

![Add Application](../images/portal-manage-access-add-application.png)

### 1.2 Associate an Entra app to a Microsoft Web IQ AppID

Each Microsoft Web IQ account resource in Azure has a billing account automatically created in the data plane. The billing account is the logical container used for managing access to the AppIDs in the billing account and scopes to filter search results.

AppIDs (or appProfiles as referred to in the API) are associated to API keys or Entra ID service principals. They control which Search API endpoints are accessible to the associated applications and have an associated QPS (queries per second). This can differ per Search API service.

An AppID is automatically created for the billing account when you create a Microsoft Web IQ account. At this time you cannot create additional AppIDs directly and instead need to open a ticket with support.

Creating new API keys or associating an Entra ID service principal to the AppID must be performed through the Microsoft Web IQ Portal. There is no documented way to do to this programmatically at this time.

In the Microsoft Web IQ Portal navigate to the API Access link and in the Authentication section select the Entra ID menu. 

![API Access Menu](../images/portal-api-access-bind-application.png)

Click the Bind Application (Client) ID and enter the Entra ID service principal client id and hit the Add button. This will grant the service principal access to use the Web IQ AppID to use the Search API.

![Bind Entra Application](../images/portal-api-access-bind-application-popup.png)

## 2. Create Microsoft Web IQ Scopes

Once the Entra ID service principal has been granted acces to the Customer Management API it can be used to create [Microsoft Web IQ Scopes](https://learn.microsoft.com/en-us/web-iq/reference/scopes). A scope in the context of Web IQ is a way to filter requests to Web IQ to restrict a search to specific sites, paths, or urls or exclude specific sites, paths, or urls from a search. Scopes can then be included in the request (as we'll see later) filter results. 

### 2.1 Retrieve authorized billing accounts and AppIDs

Creation of a scope requires the request specify a billing account because the scopes exist under the billing account. The Customer Management API has an endpoint named [authorized-resources](https://learn.microsoft.com/en-us/web-iq/reference/authorized-resources) that can be used to fetch the billing accounts and AppIDs the Entra ID service principal is able to access to. This can be used to pull the billing account id which will be included in later requests to administer scopes.

The code below enumerates the billing account that the service principal created in this notebook has access to for the Customer Management API.


In [ ]:
from azure.identity import ClientSecretCredential
import os
import requests
import json
from dotenv import load_dotenv

# Load environment variables from .env file
load_dotenv()

# Obtain an access token for the current identity context
credential = ClientSecretCredential(
    tenant_id=os.getenv("ENTRA_TENANT_ID"),
    client_id=os.getenv("WEBIQ_APP_CLIENT_ID"),
    client_secret=os.getenv("WEBIQ_APP_CLIENT_SECRET")
)
access_token = credential.get_token("https://api.microsoft.ai/.default").token

def get_webiq_authorized_accounts(token):
    url = "https://api.microsoft.ai/v1/customer/authorized-resources"
    headers = {
        "Authorization": f"Bearer {token}",
    }
    response = requests.get(url, headers=headers)
    response.raise_for_status()
    return response.json()

response = get_webiq_authorized_accounts(
    token = access_token
)
billing_account_id = response[0]["billingAccounts"][0].get("billingAccountId")
print("The billing account id is: ", billing_account_id)
print("Raw response:")
print(json.dumps(response, indent=2))

### 2.2 Create a Web IQ Scope

Now that the service principal has been granted access to the Customer Management API and we have the billing account id we can configure a scope.

In the code below we create a scope named exclude-imdb which will exclude imdb.com from searches but allow all other sites indexed by the service. We create a separate scope named include-imdb which will restrict the service to searching just imdb.com.



In [ ]:
from azure.identity import ClientSecretCredential
import requests
import json
import os
from dotenv import load_dotenv

# Load environment variables from a .env file
load_dotenv()

# Obtain an access token for the current identity context
credential = ClientSecretCredential(
    tenant_id=os.getenv("ENTRA_TENANT_ID"),
    client_id=os.getenv("WEBIQ_APP_CLIENT_ID"),
    client_secret=os.getenv("WEBIQ_APP_CLIENT_SECRET")
)
access_token = credential.get_token("https://api.microsoft.ai/.default").token

def get_webiq_scope(token: str, billing_account_id: str, scope_name):
    url = f"https://api.microsoft.ai/v1/customer/billing-accounts/{billing_account_id}/scopes/{scope_name}"
    headers = {
        "Authorization": f"Bearer {token}",
    }
    response = requests.get(url, headers=headers)
    response.raise_for_status()
    return response.json()

def create_webiq_scope(token: str, billing_account_id: str, scope_name: str, includesites: list, excludesites: list):

    try: 
        if get_webiq_scope(token, billing_account_id, scope_name):
            logger.info(f"Scope '{scope_name}' already exists.")
            return get_webiq_scope(token, billing_account_id, scope_name)
    except requests.exceptions.HTTPError as e:
        logger.info(f"Scope '{scope_name}' does not exist. Creating a new one.")
        if e.response.status_code != 404:
            raise

    url = f"https://api.microsoft.ai/v1/customer/billing-accounts/{billing_account_id}/scopes"
    headers = {
        "Authorization": f"Bearer {token}",
    }
    data = {
        "name": scope_name,
        "includeSites": includesites,
        "excludeSites": excludesites
    }
    response = requests.post(url, headers=headers, json=data)
    response.raise_for_status()
    return response.json()

# Create an exclude scope

logging.info("Creating exclude-imdb scope...")
response = create_webiq_scope(
    token = access_token,
    billing_account_id = billing_account_id,
    scope_name = "exclude-imdb",
    includesites = [],
    excludesites = ["imdb.com"]
)
print(json.dumps(response, indent=2))

logging.info("Creating include-imdb scope...")
response = create_webiq_scope(
    token = access_token,
    billing_account_id = billing_account_id,
    scope_name = "include-imdb",
    includesites = ["imdb.com"],
    excludesites = []
)
print(json.dumps(response, indent=2))

## 3. Perform a search using Web IQ using the REST API

Microsoft Web IQ offers both a REST API endpoint and an MCP Server endpoint. This section will demonstrate using the REST API. Before an application can use the APIs it must be associated to an Web IQ AppID. We did this in the first section with the service principal we created.

Microsoft provides a [Web IQ SDK](https://learn.microsoft.com/en-us/web-iq/integrations/sdk?pivots=programming-language-python), but we'll be using the requests library for demonstration purposes.

### 3.1 Search using the auto endpoint without a scope

Web IQ provides a [number of different search API endpoints](https://learn.microsoft.com/en-us/web-iq/guides/best-practices). Best practices recommend starting with the auto endpoint. Since we defined scopes earlier, the API will expect one to be passed unless we set the failOnMissingScope property to false in the body of the request.

The code below uses the auto endpoint to search for three actors in the movie The Goonies. You'll notice imdb.com results are included in the response from ther service because we haven't included our exclude scope.

In [ ]:
import os
import json
import requests
from dotenv import load_dotenv
from azure.identity import ClientSecretCredential

# Load environment variables from .env file
load_dotenv(override=True)

WEBIQ_APPID = os.getenv("WEBIQ_APPID")

def search_webiq_auto(token: str, webiq_appid: str, search_query: str):
    url = f"https://api.microsoft.ai/v3/auto"
    headers = {
        "Authorization": f"Bearer {token}",
        "content-type": "application/json"
    }

    query = {
        "APPID": webiq_appid
    }

    data = {
        "query": search_query,
        "language": "en",
        "region": "US",
        "mode": "lite",
        "maxResponseLength": 1000,
        "maxCitations": 5,
        "responseFilter": ["web", "news"],
        "failOnMissingScope": False
    }
    response = requests.post(url, headers=headers, params=query, json=data)
    response.raise_for_status()
    return response.json()


# Obtain an access token for the current identity context
credential = ClientSecretCredential(
    tenant_id=os.getenv("ENTRA_TENANT_ID"),
    client_id=os.getenv("WEBIQ_APP_CLIENT_ID"),
    client_secret=os.getenv("WEBIQ_APP_CLIENT_SECRET")
)
access_token = credential.get_token("https://api.microsoft.ai/.default").token

# Issue query
response = search_webiq_auto(
    token = access_token,
    webiq_appid = WEBIQ_APPID,
    search_query = "5 actors in The Goonies"
)

print(json.dumps(response, indent=2))

### 3.2 Search using the auto endpoint with an exclude scope

This time we'll add the exclude scope to the request. 

Notice how the results now exclude the imdb results.


In [ ]:
import os
import requests
import json
from dotenv import load_dotenv
from azure.identity import ClientSecretCredential

# Load environment variables from .env file
load_dotenv(override=True)

WEBIQ_APPID = os.getenv("WEBIQ_APPID")

def search_webiq_auto(token: str, webiq_appid: str, search_query: str, scope_name: str):
    url = f"https://api.microsoft.ai/v3/auto"
    headers = {
        "Authorization": f"Bearer {token}",
        "content-type": "application/json"
    }

    query = {
        "APPID": webiq_appid
    }

    data = {
        "query": search_query,
        "language": "en",
        "region": "US",
        "mode": "lite",
        "maxResponseLength": 1000,
        "maxCitations": 5,
        "responseFilter": ["web", "news"],
        "failOnMissingScope": True,
        "scopeName": scope_name
    }
    response = requests.post(url, headers=headers, params=query, json=data)
    response.raise_for_status()
    return response.json()


# Obtain an access token for the current identity context
credential = ClientSecretCredential(
    tenant_id=os.getenv("ENTRA_TENANT_ID"),
    client_id=os.getenv("WEBIQ_APP_CLIENT_ID"),
    client_secret=os.getenv("WEBIQ_APP_CLIENT_SECRET")
)
access_token = credential.get_token("https://api.microsoft.ai/.default").token

# Issue query
response = search_webiq_auto(
    token = access_token,
    webiq_appid = WEBIQ_APPID,
    search_query = "5 actors in The Goonies",
    scope_name = "exclude-imdb"
)

print(json.dumps(response, indent=2))

### 3.3 Search using the auto endpoint with an include scope

This time we'll add the include scope to the request. 

Notice how the results are now only sourced from imdb.com


In [ ]:
import os
import requests
import json
from dotenv import load_dotenv
from azure.identity import ClientSecretCredential

# Load environment variables from .env file
load_dotenv(override=True)

WEBIQ_APPID = os.getenv("WEBIQ_APPID")

def search_webiq_auto(token: str, webiq_appid: str, search_query: str, scope_name: str):
    url = f"https://api.microsoft.ai/v3/auto"
    headers = {
        "Authorization": f"Bearer {token}",
        "content-type": "application/json"
    }

    query = {
        "APPID": webiq_appid
    }

    data = {
        "query": search_query,
        "language": "en",
        "region": "US",
        "mode": "lite",
        "maxResponseLength": 1000,
        "maxCitations": 5,
        "responseFilter": ["web", "news"],
        "failOnMissingScope": True,
        "scopeName": scope_name
    }
    response = requests.post(url, headers=headers, params=query, json=data)
    response.raise_for_status()
    return response.json()


# Obtain an access token for the current identity context
credential = ClientSecretCredential(
    tenant_id=os.getenv("ENTRA_TENANT_ID"),
    client_id=os.getenv("WEBIQ_APP_CLIENT_ID"),
    client_secret=os.getenv("WEBIQ_APP_CLIENT_SECRET")
)
access_token = credential.get_token("https://api.microsoft.ai/.default").token

# Issue query
response = search_webiq_auto(
    token = access_token,
    webiq_appid = WEBIQ_APPID,
    search_query = "5 actors in The Goonies",
    scope_name = "include-imdb"
)

print(json.dumps(response, indent=2))

## 4. Perform a search using Web IQ using the MCP Server

Microsoft Web IQ offers both a REST API endpoint and an MCP Server endpoint. This section will demonstrate using the MCP Server. Before an application can use the APIs it must be associated to an Web IQ AppID. We did this in the first section with the service principal we created.

Microsoft provides a [Web IQ SDK](https://learn.microsoft.com/en-us/web-iq/integrations/sdk?pivots=programming-language-python), but we'll be using the requests library for demonstration purposes.

### 4.1 List the tools available from the MCP Server

At this time the auto endpoint is not available when using the MCP Server endpoint. Instead, the agent must choose the appropriate endpoint where each endpoint is exposed as tool as seen below.

In [ ]:
import os
import json
import logging
import token
from dotenv import load_dotenv
from azure.identity import ClientSecretCredential
from mcp import ClientSession
from mcp.client.streamable_http import streamable_http_client
from mcp.shared._httpx_utils import create_mcp_http_client

# Load environment variables from a .env file
load_dotenv()

async def webiq_mcp(action: str, access_token: str, tool_name: str = None, tool_args: dict = None):
    # Prepare the header with the Entra token passed to the function
    headers = {"Authorization": f"Bearer {access_token}"}

    # Create an HTTP client with the authorization header containing the Entra token
    http_client = create_mcp_http_client(headers=headers)

    # Use the HTTP client to communicate with the MCP endpoint
    async with streamable_http_client("https://api.microsoft.ai/v3/mcp", http_client=http_client) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()
            if action == "list":
                tools = await session.list_tools()
                return [tool.name for tool in tools.tools]
            return await session.call_tool(tool_name, tool_args)
        
# Obtain an access token for the current identity context
credential = ClientSecretCredential(
    tenant_id=os.getenv("ENTRA_TENANT_ID"),
    client_id=os.getenv("WEBIQ_APP_CLIENT_ID"),
    client_secret=os.getenv("WEBIQ_APP_CLIENT_SECRET")
)
access_token = credential.get_token("https://api.microsoft.ai/.default").token

# List the tools available in the MCP session
tool_names = await webiq_mcp(action="list", access_token=access_token)
print(json.dumps(tool_names, indent=2))

### 4.2 Perform a Web Search using the Web IQ MCP Server

The code below performs a web search using the MCP Server

In [ ]:
import os
import json
import logging
import token
from dotenv import load_dotenv
from azure.identity import ClientSecretCredential
from mcp import ClientSession
from mcp.client.streamable_http import streamable_http_client
from mcp.shared._httpx_utils import create_mcp_http_client

# Load environment variables from a .env file
load_dotenv()

async def webiq_mcp(action: str, access_token: str, tool_name: str = None, tool_args: dict = None):
    # Prepare the header with the Entra token passed to the function
    headers = {"Authorization": f"Bearer {access_token}"}

    # Create an HTTP client with the authorization header containing the Entra token
    http_client = create_mcp_http_client(headers=headers)

    # Use the HTTP client to communicate with the MCP endpoint
    async with streamable_http_client("https://api.microsoft.ai/v3/mcp", http_client=http_client) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()
            if action == "list":
                tools = await session.list_tools()
                return [tool.name for tool in tools.tools]
            return await session.call_tool(tool_name, tool_args)
        
# Obtain an access token for the current identity context
credential = ClientSecretCredential(
    tenant_id=os.getenv("ENTRA_TENANT_ID"),
    client_id=os.getenv("WEBIQ_APP_CLIENT_ID"),
    client_secret=os.getenv("WEBIQ_APP_CLIENT_SECRET")
)
access_token = credential.get_token("https://api.microsoft.ai/.default").token

# Run a web search as an MCP client
result = await webiq_mcp(
    action = "call",
    access_token = access_token,
    tool_name = "web",
    tool_args = {
        "query": "5 actors in The Goonies", 
        "language": "en",
        "region": "US",
        "mode": "lite",
        "maxLength": 1000,
        "maxResults": 5,
        "failOnMissingScope": False,
    }
)
print(json.dumps(json.loads(result.content[0].text), indent=2))

## 5. Perform a search using Web IQ using the MCP Server



## 5. Use the Microsoft Web IQ SDK

In the previous sections you called the REST and MCP endpoints directly. The Microsoft Web IQ Python SDK (`webiq`) wraps the same search endpoints in typed clients. It accepts the Entra credential you already created, so no extra authentication setup is needed. The SDK is pinned in `requirements.txt`.

### 5.1 Run an Auto search

Auto search returns evidence with citations instead of a list of links. The `mode` option sets how much work the service does for each request. This example uses `advanced`.

In [ ]:
from webiq import WebIQClient
from webiq.types import AutoMode

# Obtain an access token for the current identity context
credential = ClientSecretCredential(
    tenant_id=os.getenv("ENTRA_TENANT_ID"),
    client_id=os.getenv("WEBIQ_APP_CLIENT_ID"),
    client_secret=os.getenv("WEBIQ_APP_CLIENT_SECRET")
)

with WebIQClient(credential=credential) as client:
    evidence = client.auto.search(
        "What are the latest developments in retrieval-augmented generation?",
        mode=AutoMode.advanced,
        max_citations=5,
        timeout=60.0,
    )
    for result in evidence.results or []:
        print(result.evidence)
        for citation in result.citations or []:
            print(citation.title, citation.url)

### 5.2 Search the web and news

Web and news search return passages when `content_format` is set to `passage`, the recommended format for grounding.

In [ ]:
from webiq.types import ContentFormat

with WebIQClient(credential=credential) as client:
    web = client.web.search(
        "Microsoft Web IQ",
        max_results=5,
        content_format=ContentFormat.passage,
    )
    for result in web.webResults or []:
        print(result.title, result.url)

    news = client.news.search(
        "technology",
        max_results=5,
        content_format=ContentFormat.passage,
    )
    for item in news.newsResults or []:
        print(item.title, item.source)

### 5.3 Fetch a page with Browse

Browse retrieves a single page. The `query` selects the passages to return from that page.

In [ ]:
from webiq.types import BrowseContentFormat

with WebIQClient(credential=credential) as client:
    page = client.browse.fetch(
        "https://techcommunity.microsoft.com/blog/azuredevcommunityblog/why-ai-agents-need-external-grounding/4565283",
        query="How fast is Web IQ",
        content_format=BrowseContentFormat.passage,
        max_length=3000,
        live_crawl="fallback",
    )
    print(page.content)

## 6. Clean up resources

In this step you will remove the resources created in this lab so you aren't charged for them. Destroy the Terraform resources first, since the service principal is still needed until then. Delete the service principal last.

### 6.1 Destroy the Terraform resources

This removes the resource group and the Microsoft Web IQ account created in section 0. It reuses the `env` variables from the Terraform step.

In [ ]:
import subprocess

result = subprocess.run(
    ["terraform", "destroy", "-auto-approve", "-input=false", "-no-color"],
    cwd="../infra", capture_output=True, text=True,
)
print(result.stdout)
if result.returncode != 0:
    raise RuntimeError(result.stderr)

### 6.2 Delete the service principal

Deleting the Entra app registration also deletes its service principal. After this step the `WEBIQ_APP_CLIENT_ID` and `WEBIQ_APP_CLIENT_SECRET` values in `.env` no longer work.

In [ ]:
import os
import subprocess
from dotenv import load_dotenv

load_dotenv()

subprocess.run(
    ["az", "ad", "app", "delete", "--id", os.getenv("WEBIQ_APP_CLIENT_ID")],
    capture_output=True, text=True, check=True,
)
logging.info("Deleted the webiq-demo-app app registration and service principal.")